# Chapter 2 Practical 07: Context, Explainability, and Zero-Shot Demo

Learning objectives:
- Apply context-aware re-ranking.
- Explain recommendations with shared content features.
- Build a zero-shot style text search interface.
- Keep generative enrichment as an optional stub, not a required API call.

Slide connection: context-aware recommendation, explainable recommendation, zero-shot examples, and generative metadata enrichment.


Load the same movie data so this notebook connects back to the earlier practicals.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("../data")
if not (DATA_DIR / "movies_chapter2.csv").exists():
    DATA_DIR = Path("chapter_02_content_based/data")

movies = pd.read_csv(DATA_DIR / "movies_chapter2.csv")
movies.head()


,movie_id,title,genres,director,year,duration_min,rating,family_friendly,description,keywords
0,1,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,2010,148,8.8,0,A thief enters layered dreams to plant an idea...,dreams heist subconscious mind-bending
1,2,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,2014,169,8.7,0,Astronauts travel through a wormhole to find a...,space exploration wormhole survival family
2,3,Titanic,Romance|Drama,James Cameron,1997,195,7.9,0,A young couple from different social classes f...,romance ship tragedy historical
3,4,The Matrix,Sci-Fi|Action,The Wachowskis,1999,136,8.7,0,A hacker discovers that reality is a simulated...,simulation hacker reality action cyberpunk
4,5,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1995,81,8.3,1,A cowboy doll feels threatened when a space ra...,toys friendship family adventure


Create a base TF-IDF recommender. This acts as the reliable fallback for zero-shot text queries.


In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies["search_text"] = (
    movies["title"] + " " +
    movies["genres"].str.replace("|", " ", regex=False) + " " +
    movies["director"] + " " +
    movies["description"] + " " +
    movies["keywords"]
)

vectorizer = TfidfVectorizer(stop_words="english")
item_matrix = vectorizer.fit_transform(movies["search_text"])


Zero-shot style search lets the user describe what they want instead of choosing a seed item.


In [3]:
def search_movies(query, n=6):
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, item_matrix).ravel()
    results = movies[["title", "genres", "director", "family_friendly", "duration_min"]].copy()
    results["base_score"] = scores
    return results.sort_values("base_score", ascending=False).head(n)

search_movies("movies about space exploration")


,title,genres,director,family_friendly,duration_min,base_score
1,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,0,169,0.319615
10,Gravity,Sci-Fi|Thriller|Drama,Alfonso Cuaron,0,91,0.102175
4,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1,81,0.084377
7,The Martian,Sci-Fi|Adventure|Comedy,Ridley Scott,0,144,0.083270
0,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,0,148,0.000000
2,Titanic,Romance|Drama,James Cameron,0,195,0.000000


Context-aware recommendation adjusts the ranking for the current situation.


In [4]:
def rerank_for_context(results, context):
    adjusted = results.copy()
    adjusted["context_bonus"] = 0.0

    if context == "morning_mobile":
        adjusted.loc[adjusted["duration_min"] <= 110, "context_bonus"] += 0.12
    elif context == "evening_tv":
        adjusted.loc[adjusted["duration_min"] >= 120, "context_bonus"] += 0.10
    elif context == "family_mode":
        adjusted.loc[adjusted["family_friendly"] == 1, "context_bonus"] += 0.20

    adjusted["final_score"] = adjusted["base_score"] + adjusted["context_bonus"]
    return adjusted.sort_values("final_score", ascending=False)

base = search_movies("light comedy for family evening", n=8)
rerank_for_context(base, "family_mode")


,title,genres,director,family_friendly,duration_min,base_score,context_bonus,final_score
9,Paddington,Comedy|Family|Adventure,Paul King,1,95,0.504052,0.2,0.704052
4,Toy Story,Animation|Adventure|Comedy|Family,John Lasseter,1,81,0.324569,0.2,0.524569
5,Finding Nemo,Animation|Adventure|Family,Andrew Stanton,1,100,0.204056,0.2,0.404056
7,The Martian,Sci-Fi|Adventure|Comedy,Ridley Scott,0,144,0.122664,0.0,0.122664
1,Interstellar,Sci-Fi|Adventure|Drama,Christopher Nolan,0,169,0.120162,0.0,0.120162
0,Inception,Sci-Fi|Thriller|Action,Christopher Nolan,0,148,0.000000,0.0,0.000000
2,Titanic,Romance|Drama,James Cameron,0,195,0.000000,0.0,0.000000
3,The Matrix,Sci-Fi|Action,The Wachowskis,0,136,0.000000,0.0,0.000000


Explanations should be short and specific. Here we explain by shared genres, director, and high-weight query terms.


In [5]:
def explain_with_features(query, title, top_terms=5):
    movie = movies[movies["title"].eq(title)].iloc[0]
    query_vector = vectorizer.transform([query]).toarray().ravel()
    movie_vector = item_matrix[movies.index[movies["title"].eq(title)][0]].toarray().ravel()
    contribution = query_vector * movie_vector
    terms = vectorizer.get_feature_names_out()
    best_terms = [terms[i] for i in contribution.argsort()[::-1][:top_terms] if contribution[i] > 0]

    return {
        "movie": title,
        "recommended_because_it_shares": ", ".join(best_terms) if best_terms else "related content features",
        "genres": movie["genres"],
        "director": movie["director"],
    }

query = "space survival astronaut"
top_title = search_movies(query, n=1).iloc[0]["title"]
explain_with_features(query, top_title)


{'movie': 'The Martian',
 'recommended_because_it_shares': 'astronaut, survival, space',
 'genres': 'Sci-Fi|Adventure|Comedy',
 'director': 'Ridley Scott'}

A contribution table makes the explanation inspectable rather than magical.


In [6]:
def contribution_table(query, title):
    idx = movies.index[movies["title"].eq(title)][0]
    q = vectorizer.transform([query]).toarray().ravel()
    x = item_matrix[idx].toarray().ravel()
    terms = vectorizer.get_feature_names_out()
    table = pd.DataFrame({"term": terms, "query_weight": q, "movie_weight": x})
    table["contribution"] = table["query_weight"] * table["movie_weight"]
    return table[table["contribution"] > 0].sort_values("contribution", ascending=False).head(10)

contribution_table("space survival astronaut", top_title)


,term,query_weight,movie_weight,contribution
9,astronaut,0.644293,0.373186,0.240442
134,survival,0.569141,0.164828,0.093811
126,space,0.510848,0.147946,0.075578


Optional SBERT can replace TF-IDF for zero-shot search if it is available.


In [7]:
semantic_search_available = False
try:
    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    embeddings = model.encode(movies["search_text"].tolist(), show_progress_bar=False)
    semantic_search_available = True
except Exception as exc:
    print("Semantic search model is optional and not available here. TF-IDF search remains active.")
    print(type(exc).__name__, str(exc)[:160])

semantic_search_available


/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:246: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/Users/mehrdadjalali/Library/Python/3.9/lib/python/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google 

True

Generative enrichment can be introduced as a stub. Students should not need an API key to run the notebook.


In [8]:
def enrich_metadata_stub(title):
    return {
        "title": title,
        "possible_extra_tags": ["teaching stub", "replace with reviewed metadata", "no API call required"],
        "note": "In production, generated metadata should be checked before it affects recommendations.",
    }

enrich_metadata_stub("Interstellar")


{'title': 'Interstellar',
 'possible_extra_tags': ['teaching stub',
  'replace with reviewed metadata',
  'no API call required'],
 'note': 'In production, generated metadata should be checked before it affects recommendations.'}

## What did we learn?

- Context can re-rank otherwise reasonable recommendations.
- Explanations should name concrete shared features.
- Zero-shot search can be taught with TF-IDF first and upgraded to embeddings when available.
- Generative enrichment is powerful, but it should be optional and reviewed.

Exercises:
1. Add a `late_night` context and define your own re-ranking rule.
2. Write two natural-language queries and compare their recommendation lists.
